# Movie Recommender System

## Notebook structure
1. Introduction
2. Configuration
3. Data Loading
4. Data Analysis
5. Data Preparation
6. User-Based Collaborative Filtering
7. Matrix Factorization (SVD)
8. Model Evaluation (RMSE, Precision@10, Recall@10)
9. Model Selection
10. Generating Recommendations
11. Cold-Start Strategy
12. Final Recommendations
13. Readable Recommendations (Movie Titles)
14. Conclusion

## Usage

In the **Task Runner** cell at the bottom, We can run each part by setting one of the flags true.


## 1. Introduction

In this notebook, I compare two recommendation approaches:
- **User-Based Collaborative Filtering**
- **Matrix Factorization (SVD)**

Then I select the best model and generate **Top-10 recommendations** for the users in `ratings_test.csv`. I also use a popularity-based strategy for **cold-start users**.


## 2. Configuration

In [1]:
from pathlib import Path

# I configured these paths so I can run the notebook from inside the 'notebooks/' folder.
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = (NOTEBOOK_DIR / "../data").resolve()
OUTPUT_DIR = (NOTEBOOK_DIR / "../output").resolve()

MOVIES_PATH = DATA_DIR / "movies.csv"
RATINGS_TRAIN_PATH = DATA_DIR / "ratings_train.csv"
RATINGS_TEST_PATH = DATA_DIR / "ratings_test.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

DATA_DIR: C:\Users\Admin\Storage\Akademik\Dersler\Erasmus Classes\Data Mining\Assignment2\recommender-assignment\data
OUTPUT_DIR: C:\Users\Admin\Storage\Akademik\Dersler\Erasmus Classes\Data Mining\Assignment2\recommender-assignment\output


## 3. Data Loading

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from collections import defaultdict

from surprise import Dataset, Reader, KNNBasic, SVD
from surprise.model_selection import cross_validate, train_test_split

In [3]:
def load_datasets():
    """Load all datasets used in the assignment."""
    movies = pd.read_csv(MOVIES_PATH)
    ratings_train = pd.read_csv(RATINGS_TRAIN_PATH)
    ratings_test = pd.read_csv(RATINGS_TEST_PATH)
    return movies, ratings_train, ratings_test


movies, ratings_train, ratings_test = load_datasets()

print("movies shape:", movies.shape)
print("ratings_train shape:", ratings_train.shape)
print("ratings_test shape:", ratings_test.shape)

display(movies.head())
display(ratings_train.head())
display(ratings_test.head())

movies shape: (9742, 3)
ratings_train shape: (97801, 4)
ratings_test shape: (100, 11)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


,userId,recommendation1,recommendation2,recommendation3,recommendation4,recommendation5,recommendation6,recommendation7,recommendation8,recommendation9,recommendation10
0,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Data Analysis

In [4]:
def analyze_dataset(movies, ratings_train, ratings_test):
    """Run the exploratory checks used in the report."""

    print("Missing values in movies:")
    print(movies.isna().sum())
    print()

    print("Missing values in ratings_train:")
    print(ratings_train.isna().sum())
    print()

    print("Missing values in ratings_test:")
    print(ratings_test.isna().sum())
    print()

    print("Unique users in train:", ratings_train["userId"].nunique())
    print("Unique movies in train:", ratings_train["movieId"].nunique())
    print("Rating min:", ratings_train["rating"].min())
    print("Rating max:", ratings_train["rating"].max())
    print()

    print("Rating distribution:")
    print(ratings_train["rating"].value_counts().sort_index())
    print()

    duplicate_count = ratings_train.duplicated(subset=["userId", "movieId"]).sum()
    print("Duplicate user-movie pairs:", duplicate_count)
    print()

    train_users = set(ratings_train["userId"].unique())
    test_users = set(ratings_test["userId"].unique())
    cold_start_users = sorted(test_users - train_users)

    print("Number of test users:", len(test_users))
    print("Number of cold-start users:", len(cold_start_users))
    print("Cold-start user IDs:", cold_start_users)
    print()

    user_activity = ratings_train.groupby("userId")["movieId"].count()
    movie_popularity = ratings_train.groupby("movieId")["userId"].count()

    print("User activity summary:")
    print(user_activity.describe())
    print()

    print("Movie popularity summary:")
    print(movie_popularity.describe())

    plt.figure(figsize=(6, 4))
    ratings_train["rating"].hist(bins=10)
    plt.xlabel("Rating")
    plt.ylabel("Frequency")
    plt.title("Distribution of ratings in training set")
    plt.show()

## 5. Data Preparation

In [5]:
def prepare_data(movies, ratings_train, ratings_test):
    """Prepare reusable objects for modelling and recommendation."""

    # Only keep information that is actually used later.
    all_movie_ids = set(movies["movieId"].unique())
    train_users = set(ratings_train["userId"].unique())
    test_users = set(ratings_test["userId"].unique())
    cold_start_users = sorted(test_users - train_users)

    user_seen_movies = ratings_train.groupby("userId")["movieId"].apply(set).to_dict()

    movie_id_to_title = dict(zip(movies["movieId"], movies["title"]))
    movie_id_to_genres = dict(zip(movies["movieId"], movies["genres"]))

    reader = Reader(rating_scale=(0.5, 5.0))
    surprise_data = Dataset.load_from_df(
        ratings_train[["userId", "movieId", "rating"]],
        reader
    )

    return {
        "all_movie_ids": all_movie_ids,
        "train_users": train_users,
        "test_users": test_users,
        "cold_start_users": cold_start_users,
        "user_seen_movies": user_seen_movies,
        "movie_id_to_title": movie_id_to_title,
        "movie_id_to_genres": movie_id_to_genres,
        "reader": reader,
        "surprise_data": surprise_data,
    }


prepared = prepare_data(movies, ratings_train, ratings_test)

print("Prepared objects:")
print("- Number of all movies:", len(prepared["all_movie_ids"]))
print("- Number of train users:", len(prepared["train_users"]))
print("- Number of cold-start users:", len(prepared["cold_start_users"]))

Prepared objects:
- Number of all movies: 9742
- Number of train users: 600
- Number of cold-start users: 10


## 6. User-Based Collaborative Filtering

In [6]:
def build_user_based_model():
    """Create the user-based collaborative filtering model."""
    sim_options = {
        "name": "pearson",
        "user_based": True,
    }
    return KNNBasic(sim_options=sim_options, verbose=False)

## 7. Matrix Factorization (SVD)

In [7]:
def build_svd_model():
    """Create the SVD model used for matrix factorization."""
    return SVD(random_state=42)

## 8. Model Evaluation (RMSE, Precision@10, Recall@10)

In [8]:
def evaluate_rmse(data):
    """Evaluate both models with 5-fold cross-validation using RMSE."""
    user_model = build_user_based_model()
    svd_model = build_svd_model()

    print("Evaluating User-Based Collaborative Filtering...")
    results_user = cross_validate(
        user_model,
        data,
        measures=["RMSE"],
        cv=5,
        verbose=True
    )

    print("\nEvaluating SVD...")
    results_svd = cross_validate(
        svd_model,
        data,
        measures=["RMSE"],
        cv=5,
        verbose=True
    )

    rmse_user = float(np.mean(results_user["test_rmse"]))
    rmse_svd = float(np.mean(results_svd["test_rmse"]))

    rmse_table = pd.DataFrame(
        {
            "Model": ["User-Based CF", "SVD"],
            "RMSE": [rmse_user, rmse_svd],
        }
    )

    return rmse_table

In [9]:
def precision_recall_at_k(model, trainset, testset, all_movie_ids, k=10, threshold=4.0):
    """Compute Precision@K and Recall@K for a Surprise model."""

    model.fit(trainset)

    train_user_items = defaultdict(set)
    for inner_uid, inner_iid, _ in trainset.all_ratings():
        raw_uid = trainset.to_raw_uid(inner_uid)
        raw_iid = trainset.to_raw_iid(inner_iid)
        train_user_items[raw_uid].add(raw_iid)

    test_user_ratings = defaultdict(list)
    for uid, iid, true_rating in testset:
        test_user_ratings[uid].append((iid, true_rating))

    precisions = []
    recalls = []

    for user, ratings in test_user_ratings.items():
        seen_movies = train_user_items[user]
        unseen_movies = all_movie_ids - seen_movies

        predictions = []
        for movie_id in unseen_movies:
            pred = model.predict(user, movie_id)
            predictions.append((movie_id, pred.est))

        predictions.sort(key=lambda x: x[1], reverse=True)
        top_k_movie_ids = [movie_id for movie_id, _ in predictions[:k]]

        relevant_movies = [iid for iid, rating in ratings if rating >= threshold]
        if len(relevant_movies) == 0:
            continue

        hits = sum(1 for movie_id in top_k_movie_ids if movie_id in relevant_movies)

        precisions.append(hits / k)
        recalls.append(hits / len(relevant_movies))

    return float(np.mean(precisions)), float(np.mean(recalls))

In [10]:
def compare_models(data, all_movie_ids, k=10, threshold=4.0):
    """Compare User-Based CF and SVD using RMSE, Precision@10, and Recall@10."""

    rmse_table = evaluate_rmse(data)

    trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

    user_model = build_user_based_model()
    svd_model = build_svd_model()

    precision_user, recall_user = precision_recall_at_k(
        user_model, trainset, testset, all_movie_ids, k=k, threshold=threshold
    )
    precision_svd, recall_svd = precision_recall_at_k(
        svd_model, trainset, testset, all_movie_ids, k=k, threshold=threshold
    )

    ranking_table = pd.DataFrame(
        {
            "Model": ["User-Based CF", "SVD"],
            "Precision@10": [precision_user, precision_svd],
            "Recall@10": [recall_user, recall_svd],
        }
    )

    comparison_table = rmse_table.merge(ranking_table, on="Model")
    comparison_table = comparison_table.sort_values(by=["RMSE", "Precision@10"], ascending=[True, False]).reset_index(drop=True)

    print("Model comparison:")
    display(comparison_table)

    return comparison_table

## 9. Model Selection

In [11]:
def select_best_model(comparison_table):
    """Select the best model. RMSE is primary, ranking metrics are used for support."""
    best_model_name = comparison_table.iloc[0]["Model"]

    if best_model_name == "SVD":
        best_model = build_svd_model()
    else:
        best_model = build_user_based_model()

    print("Selected final model:", best_model_name)
    return best_model_name, best_model

## 10. Generating Recommendations

In [12]:
def fit_final_model(model, data):
    """Fit the final model on the full training data."""
    full_trainset = data.build_full_trainset()
    model.fit(full_trainset)
    return model


def recommend_top_n(model, user_id, user_seen_movies, all_movie_ids, n=10):
    """Recommend Top-N unseen movies for a warm user."""
    seen_movies = user_seen_movies.get(user_id, set())
    unseen_movies = all_movie_ids - seen_movies

    predictions = []
    for movie_id in unseen_movies:
        pred = model.predict(user_id, movie_id)
        predictions.append((movie_id, pred.est))

    predictions.sort(key=lambda x: x[1], reverse=True)
    return [movie_id for movie_id, _ in predictions[:n]]

## 11. Cold-Start Strategy

In [13]:
def build_cold_start_recommendations(ratings_train, min_ratings=20, n=10):
    """Create a popularity-based fallback list for cold-start users."""
    movie_stats = ratings_train.groupby("movieId").agg(
        mean_rating=("rating", "mean"),
        rating_count=("rating", "count")
    ).reset_index()

    popular_movies = movie_stats[movie_stats["rating_count"] >= min_ratings].copy()
    popular_movies = popular_movies.sort_values(
        by=["mean_rating", "rating_count"],
        ascending=[False, False]
    )

    return popular_movies["movieId"].head(n).tolist()

## 12. Final Recommendations

In [14]:
def generate_submission_file(final_model, ratings_test, train_users, user_seen_movies, all_movie_ids, cold_start_top10):
    """Generate the final submission file for ratings_test.csv."""
    output = ratings_test.copy()

    for idx, row in output.iterrows():
        user_id = row["userId"]

        if user_id in train_users:
            recs = recommend_top_n(final_model, user_id, user_seen_movies, all_movie_ids, n=10)
        else:
            recs = cold_start_top10

        for i in range(10):
            output.at[idx, f"recommendation{i+1}"] = recs[i]

    return output


def validate_submission(output, all_movie_ids):
    """Validate the generated submission file."""
    print("Output shape:", output.shape)
    print()

    print("Missing values per column:")
    print(output.isna().sum())
    print()

    recommended_movies = set()
    for col in output.columns[1:]:
        recommended_movies.update(output[col].unique())

    invalid_movie_ids = recommended_movies - all_movie_ids
    print("Invalid movie IDs:", invalid_movie_ids)
    print()

    def has_duplicates(row):
        recs = row[1:]
        return len(recs) != len(set(recs))

    duplicate_rows = int(output.apply(has_duplicates, axis=1).sum())
    print("Users with duplicate recommendations:", duplicate_rows)

## 13. Readable Recommendations (Movie Titles)

In [15]:
def make_readable_recommendations(output, movie_id_to_title, movie_id_to_genres):
    """Create human-readable recommendation tables using movie titles and genres."""

    title_version = output.copy()
    for col in title_version.columns[1:]:
        title_version[col] = title_version[col].map(movie_id_to_title)

    readable_rows = []
    for _, row in output.iterrows():
        user_id = row["userId"]
        movie_ids = row[1:].tolist()

        formatted_movies = []
        for movie_id in movie_ids:
            title = movie_id_to_title.get(movie_id, f"Unknown movie ({movie_id})")
            genres = movie_id_to_genres.get(movie_id, "Unknown genres")
            formatted_movies.append(f"{title} [{genres}]")

        readable_rows.append(
            {
                "userId": user_id,
                "Recommended Movies": " | ".join(formatted_movies)
            }
        )

    readable_df = pd.DataFrame(readable_rows)
    return title_version, readable_df


def show_example_user(output, movie_id_to_title, movie_id_to_genres, user_id=None):
    """Display readable recommendations for one user."""
    if user_id is None:
        user_id = int(output.iloc[0]["userId"])

    row = output[output["userId"] == user_id].iloc[0]
    print(f"Recommendations for user {user_id}:")
    for movie_id in row.iloc[1:].tolist():
        print(f"- {movie_id_to_title[movie_id]} | {movie_id_to_genres[movie_id]}")

## 14. Conclusion

Based on the results in this notebook, I select the best model and generate both the submission file and a human-readable recommendation file. This gives me both the assignment output and a readable result


## Task Runner

The cell below is the control center of this notebook. To run a specific task, We only need to set one of the flags below true.


In [16]:
# -------------------------------------------------------------------
# TASK RUNNER
# Set only ONE task to True
# -------------------------------------------------------------------

run_analysis = False
run_model_comparison = False
run_generate_submission = True
run_readable_output = True


if run_analysis:
    analyze_dataset(movies, ratings_train, ratings_test)

if run_model_comparison:
    comparison_table = compare_models(prepared["surprise_data"], prepared["all_movie_ids"])
    display(comparison_table)

if run_generate_submission:
    final_model = fit_final_model(build_svd_model(), prepared["surprise_data"])
    cold_start_top10 = build_cold_start_recommendations(ratings_train)

    output = generate_submission_file(
        final_model,
        ratings_test,
        prepared["train_users"],
        prepared["user_seen_movies"],
        prepared["all_movie_ids"],
        cold_start_top10,
    )

    validate_submission(output, prepared["all_movie_ids"])
    output.to_csv(OUTPUT_DIR / "ratings_test_filled.csv", index=False)
    print("Submission file saved.")

if run_readable_output:
    output = pd.read_csv(OUTPUT_DIR / "ratings_test_filled.csv")

    title_output, readable_df = make_readable_recommendations(
        output,
        prepared["movie_id_to_title"],
        prepared["movie_id_to_genres"],
    )

    display(title_output.head())
    display(readable_df.head())
    readable_df.to_csv(OUTPUT_DIR / "recommendations_readable.csv", index=False)
    show_example_user(output, prepared["movie_id_to_title"], prepared["movie_id_to_genres"])

Output shape: (100, 11)

Missing values per column:
userId              0
recommendation1     0
recommendation2     0
recommendation3     0
recommendation4     0
recommendation5     0
recommendation6     0
recommendation7     0
recommendation8     0
recommendation9     0
recommendation10    0
dtype: int64

Invalid movie IDs: set()

Users with duplicate recommendations: 0
Submission file saved.


,userId,recommendation1,recommendation2,recommendation3,recommendation4,recommendation5,recommendation6,recommendation7,recommendation8,recommendation9,recommendation10
0,3,Groundhog Day (1993),Goodfellas (1990),Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mr. Holland's Opus (1995),K-PAX (2001),Finding Nemo (2003),Hoop Dreams (1994),"Last Detail, The (1973)","Dark Knight, The (2008)",Gaslight (1944)
1,7,"Godfather, The (1972)","Godfather: Part II, The (1974)",All About Eve (1950),12 Angry Men (1957),Raiders of the Lost Ark (Indiana Jones and the...,Toy Story 3 (2010),Whiplash (2014),North by Northwest (1959),Schindler's List (1993),"Shawshank Redemption, The (1994)"
2,11,"Matrix, The (1999)",Star Wars: Episode IV - A New Hope (1977),Lawrence of Arabia (1962),Raiders of the Lost Ark (Indiana Jones and the...,Monty Python and the Holy Grail (1975),Goodfellas (1990),Singin' in the Rain (1952),"Godfather, The (1972)",Casablanca (1942),Dr. Strangelove or: How I Learned to Stop Worr...
3,25,"Usual Suspects, The (1995)",Pulp Fiction (1994),"Shawshank Redemption, The (1994)",Forrest Gump (1994),Fargo (1996),Dr. Strangelove or: How I Learned to Stop Worr...,"Godfather, The (1972)","Philadelphia Story, The (1940)",North by Northwest (1959),Casablanca (1942)
4,30,"Usual Suspects, The (1995)",Dr. Strangelove or: How I Learned to Stop Worr...,"Godfather, The (1972)","Princess Bride, The (1987)",Amadeus (1984),"Shining, The (1980)",American History X (1998),Fight Club (1999),"Green Mile, The (1999)",Double Indemnity (1944)


,userId,Recommended Movies
0,3.0,Groundhog Day (1993) [Comedy|Fantasy|Romance] ...
1,7.0,"Godfather, The (1972) [Crime|Drama] | Godfathe..."
2,11.0,"Matrix, The (1999) [Action|Sci-Fi|Thriller] | ..."
3,25.0,"Usual Suspects, The (1995) [Crime|Mystery|Thri..."
4,30.0,"Usual Suspects, The (1995) [Crime|Mystery|Thri..."


Recommendations for user 3:
- Groundhog Day (1993) | Comedy|Fantasy|Romance
- Goodfellas (1990) | Crime|Drama
- Twelve Monkeys (a.k.a. 12 Monkeys) (1995) | Mystery|Sci-Fi|Thriller
- Mr. Holland's Opus (1995) | Drama
- K-PAX (2001) | Drama|Fantasy|Mystery|Sci-Fi
- Finding Nemo (2003) | Adventure|Animation|Children|Comedy
- Hoop Dreams (1994) | Documentary
- Last Detail, The (1973) | Comedy|Drama
- Dark Knight, The (2008) | Action|Crime|Drama|IMAX
- Gaslight (1944) | Drama|Thriller
